# Week 4 — Options Extensions + Bayesian Mini-Project
**IEMS 302 consolidation project**

This week **extends the Week 3 pricer** (it imports `pricer.py` rather than
rewriting it) with three things, plus a separate Bayesian exercise:

1. **Asian options** — payoff on the *average* price; no Black-Scholes formula.
2. **Antithetic variates** — a tighter estimate for the same number of paths.
3. **Greeks** — Delta and Vega by finite differences, validated against BS.
4. **Bayesian updater** — P(up tomorrow) that sharpens as data arrives.

Run top to bottom.

## 0. Setup (imports the validated Week 3 engine)

In [ ]:
import numpy as np
import options_extensions as ext
from IPython.display import Image, display

S0, r, sigma, T, K = 180.0, 0.04, 0.28, 1.0, 180
rng = np.random.default_rng(7)

## 1. Asian options — the payoff that needs the whole path

A European call only looked at the final price $S_T$. An **arithmetic Asian
call** pays on the *average* price along the path:
$$\text{payoff} = \max\!\big(\bar S - K,\ 0\big),\qquad
\bar S = \frac{1}{m}\sum_{i=1}^{m} S_{t_i}.$$
Averaging smooths out the extremes, so an Asian option is **worth less** than the
otherwise-identical European one. Crucially, it has **no clean closed-form
price** — so we rely entirely on the method validated in Week 3.

In [ ]:
asian = ext.mc_price_asian(S0, K, r, sigma, T, 200_000, 252, "call", rng)
euro  = ext.mc_price(S0, K, r, sigma, T, 200_000, "call", rng)
print(f"Asian call    = {asian['price']:.4f}  CI [{asian['ci_low']:.4f}, {asian['ci_high']:.4f}]")
print(f"European call = {euro['price']:.4f}")
print("Asian < European?", asian['price'] < euro['price'])

## 2. Antithetic variates — cheaper accuracy

Week 3 showed error $\propto 1/\sqrt{n}$: halving it costs 4x the paths. Trick:
for every draw $Z$, also use $-Z$. The mirrored payoffs are negatively
correlated, so averaging within each pair cancels noise. Same number of paths,
smaller variance.

In [ ]:
rows = ext.antithetic_comparison(S0, K, r, sigma, T, "call", seed=7)
print(f"{'paths':>9} {'SE plain':>10} {'SE anti':>10} {'var.reduction':>14}")
for n, sp, sa, vr in rows:
    print(f"{n:>9} {sp:>10.5f} {sa:>10.5f} {vr:>13.2f}x")

In [ ]:
# efficiency must not cost correctness: BS still inside the antithetic CI
bs = ext.black_scholes(S0, K, r, sigma, T, "call")
anti = ext.mc_price_antithetic(S0, K, r, sigma, T, 1_000_000, "call", np.random.default_rng(3))
print(f"BS={bs:.4f} inside antithetic CI [{anti['ci_low']:.4f},{anti['ci_high']:.4f}]?",
      anti['ci_low'] <= bs <= anti['ci_high'])
ext._plot_antithetic(rows, "results"); display(Image("results/antithetic.png"))

## 3. Greeks — sensitivities for hedging

- **Delta** $=\partial \text{price}/\partial S_0$ : how the option moves with the stock.
- **Vega**  $=\partial \text{price}/\partial \sigma$ : how it moves with volatility.

Computed by central finite differences,
$g'(x)\approx \frac{g(x+h)-g(x-h)}{2h}$, using **common random numbers** (the same
$Z$ for both bumps) so the Monte Carlo noise cancels. Validate against the
closed-form BS Greuks: $\Delta_{\text{call}}=N(d_1)$, $\text{Vega}=S_0\sqrt{T}\,\phi(d_1)$.

In [ ]:
d_mc = ext.mc_delta(S0, K, r, sigma, T, 1_000_000, "call", rng=np.random.default_rng(5))
v_mc = ext.mc_vega (S0, K, r, sigma, T, 1_000_000, "call", rng=np.random.default_rng(6))
g = ext.bs_greeks(S0, K, r, sigma, T, "call")
print(f"Delta: MC {d_mc:.4f}  vs BS {g['delta']:.4f}")
print(f"Vega : MC {v_mc:.4f} vs BS {g['vega']:.4f}")
ext._plot_greeks(S0, K, r, sigma, T, "results", rng); display(Image("results/greeks.png"))

## 4. Bayesian updater — belief that sharpens with data

Estimate $p=P(\text{up tomorrow})$. Put a **Beta prior** on it, treat each day as
Bernoulli, and use conjugacy:
$$p\sim\text{Beta}(\alpha_0,\beta_0),\qquad
\text{posterior}=\text{Beta}(\alpha_0+\#\text{up},\ \beta_0+\#\text{down}).$$
With little data the posterior is wide (we're unsure); as days pile up it narrows
toward the truth — Bayes' Rule turning prior + evidence into belief, and Week 1's
lesson that a single number hides how confident you should be.

In [ ]:
try:
    outcomes = ext.load_up_down("GOOGL", 2); src = "live GOOGL"
except Exception:
    outcomes = (np.random.default_rng(0).random(300) < 0.53).astype(int); src = "synthetic (offline)"
post = ext.bayesian_update(outcomes, a0=1.0, b0=1.0)
print(f"source: {src}, {len(outcomes)} days")
print(f"final P(up) = {post['mean'][-1]:.4f}  CI [{post['ci_low'][-1]:.4f},{post['ci_high'][-1]:.4f}]")
ext._plot_bayes(outcomes, post, "results"); display(Image("results/bayes_posterior.png"))

## 5. Conclusion
- **Asian options** priced with no formula — the Week 3 validation is what lets
  us trust this.
- **Antithetic variates** cut variance ~1.6x for free, without breaking accuracy.
- **Finite-difference Greeks** match Black-Scholes to ~3 decimals (common random
  numbers make this possible).
- **Bayesian posterior** narrows as data arrives — uncertainty quantified, not
  hidden.

**So what (bridge to Phase 2):** the pricer now needs lots of clean, repeatable
data — many tickers, many days, fed reliably. That is a *data-engineering*
problem, which is exactly what CS 217 (Weeks 5-8) builds: a normalised database
and an automated pipeline to feed models like these.